In [2]:
from __future__ import annotations

# ============================================================
# 0. IMPORTS
# ============================================================

from collections import Counter
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
import mesa
from mesa.datacollection import DataCollector


# ============================================================
# 1. CLASE DEL AGENTE
# ============================================================

class PolicyAgent(mesa.Agent):
    """
    Agente que elige una política pública bajo racionalidad limitada.

    Heurísticas:
    1) satisficing
    2) imitación del vecino más exitoso
    """

    def __init__(
        self,
        model: "PolicyDecisionModel",
        node_id: int,
        alpha: float,
        aspiration: float,
        initial_policy: int,
    ):
        super().__init__(model)

        self.node_id = node_id
        self.alpha = float(alpha)
        self.aspiration = float(aspiration)
        self.policy = int(initial_policy)
        self.utility = 0.0

        # Variables auxiliares para actualización simultánea
        self.next_policy = self.policy
        self.next_aspiration = self.aspiration

    # -----------------------------------------------------------------
    # Información local
    # -----------------------------------------------------------------
    def neighbors(self) -> list["PolicyAgent"]:
        return [self.model.node_to_agent[j] for j in self.model.graph.neighbors(self.node_id)]

    def local_support(self, policy: int) -> float:
        """
        Proporción de vecinos que apoyan actualmente la política dada.
        """
        neigh = self.neighbors()
        if not neigh:
            return 0.0

        count = sum(1 for agent in neigh if agent.policy == policy)
        return count / len(neigh)

    # -----------------------------------------------------------------
    # Utilidad
    # -----------------------------------------------------------------
    def evaluate_policy(self, policy: int) -> float:
        """
        u_hat_i(p,t) = alpha_i * q_p + beta * s_ip(t) - gamma * c_p
        """
        q_p = self.model.policy_quality[policy]
        c_p = self.model.policy_cost[policy]
        s_ip = self.local_support(policy)

        return (self.alpha * q_p) + (self.model.beta * s_ip) - (self.model.gamma * c_p)

    def update_utility(self) -> None:
        self.utility = self.evaluate_policy(self.policy)

    # -----------------------------------------------------------------
    # Satisficing
    # -----------------------------------------------------------------
    def candidate_order(self) -> list[int]:
        """
        Orden secuencial de exploración de políticas.
        """
        policies = self.model.policies.copy()
        self.random.shuffle(policies)

        if self.model.max_policy_checks is None:
            return policies
        return policies[: self.model.max_policy_checks]

    def first_satisficing_policy(self) -> tuple[int | None, float | None]:
        """
        Devuelve la primera política que satisface el umbral de aspiración.
        """
        for p in self.candidate_order():
            u_p = self.evaluate_policy(p)
            if u_p >= self.aspiration:
                return p, u_p
        return None, None

    # -----------------------------------------------------------------
    # Imitación
    # -----------------------------------------------------------------
    def best_neighbor(self) -> "PolicyAgent | None":
        """
        Vecino con mayor utilidad observada.
        """
        neigh = self.neighbors()
        if not neigh:
            return None

        best_u = max(agent.utility for agent in neigh)
        best_agents = [agent for agent in neigh if agent.utility == best_u]
        return self.random.choice(best_agents)

    # -----------------------------------------------------------------
    # Dinámica temporal
    # -----------------------------------------------------------------
    def compute_next_state(self) -> None:
        """
        Regla híbrida:
        1) si la política actual satisface el umbral, permanece;
        2) si no satisface, busca una política satisfactoria;
        3) si no encuentra ninguna, imita al vecino más exitoso.
        """
        current_u = self.evaluate_policy(self.policy)
        current_a = self.aspiration

        chosen_policy = self.policy

        if current_u < current_a:
            p_sat, _ = self.first_satisficing_policy()

            if p_sat is not None:
                chosen_policy = p_sat
            else:
                best = self.best_neighbor()
                if best is not None and best.utility > current_u:
                    chosen_policy = best.policy
                else:
                    chosen_policy = self.policy

        self.next_policy = chosen_policy

        # Actualización del umbral de aspiración
        self.next_aspiration = ((1.0 - self.model.rho) * current_a) + (self.model.rho * current_u)

    def advance(self) -> None:
        self.policy = self.next_policy
        self.aspiration = self.next_aspiration


# ============================================================
# 2. CLASE DEL MODELO
# ============================================================

class PolicyDecisionModel(mesa.Model):
    """
    Modelo ABM de decisión sobre políticas públicas con racionalidad limitada.
    """

    def __init__(
        self,
        n_agents: int = 100,
        num_policies: int = 10,
        avg_degree: int = 6,
        rewire_prob: float = 0.10,
        beta: float = 0.80,
        gamma: float = 0.40,
        rho: float = 0.20,
        alpha_mean: float = 1.00,
        alpha_sd: float = 0.10,
        aspiration_mean: float = 0.55,
        aspiration_sd: float = 0.08,
        max_policy_checks: int | None = None,
        policy_quality: list[float] | np.ndarray | None = None,
        policy_cost: list[float] | np.ndarray | None = None,
        rng=None,
    ):
        super().__init__(rng=rng)

        # Generador NumPy explícito para robustez
        self.nprng = np.random.default_rng(rng)

        if avg_degree % 2 != 0:
            raise ValueError("En Watts-Strogatz, 'avg_degree' debe ser un entero par.")
        if num_policies < 2:
            raise ValueError("'num_policies' debe ser al menos 2.")

        self.n_agents = int(n_agents)
        self.num_policies = int(num_policies)
        self.policies = list(range(1, self.num_policies + 1))
        self.beta = float(beta)
        self.gamma = float(gamma)
        self.rho = float(rho)
        self.max_policy_checks = max_policy_checks

        # -------------------------------------------------------------
        # Atributos de las políticas
        # -------------------------------------------------------------
        if policy_quality is None:
            q = self.nprng.uniform(0.40, 0.95, size=self.num_policies)
        else:
            q = np.asarray(policy_quality, dtype=float)

        if policy_cost is None:
            c = self.nprng.uniform(0.10, 0.70, size=self.num_policies)
        else:
            c = np.asarray(policy_cost, dtype=float)

        if len(q) != self.num_policies or len(c) != self.num_policies:
            raise ValueError("policy_quality y policy_cost deben tener longitud 'num_policies'.")

        self.policy_quality = {p: float(q[p - 1]) for p in self.policies}
        self.policy_cost = {p: float(c[p - 1]) for p in self.policies}

        # -------------------------------------------------------------
        # Red de interacción
        # -------------------------------------------------------------
        graph_seed = int(self.nprng.integers(0, np.iinfo(np.int32).max))
        self.graph = nx.connected_watts_strogatz_graph(
            n=self.n_agents,
            k=int(avg_degree),
            p=float(rewire_prob),
            tries=200,
            seed=graph_seed,
        )

        # -------------------------------------------------------------
        # Creación de agentes
        # -------------------------------------------------------------
        self.node_to_agent: dict[int, PolicyAgent] = {}

        alpha_values = np.clip(
            self.nprng.normal(alpha_mean, alpha_sd, size=self.n_agents),
            0.10,
            None,
        )

        aspiration_values = np.clip(
            self.nprng.normal(aspiration_mean, aspiration_sd, size=self.n_agents),
            0.0,
            1.5,
        )

        initial_policies = self.nprng.integers(1, self.num_policies + 1, size=self.n_agents)

        for node_id, alpha_i, aspiration_i, policy_i in zip(
            self.graph.nodes(),
            alpha_values,
            aspiration_values,
            initial_policies,
        ):
            agent = PolicyAgent(
                model=self,
                node_id=int(node_id),
                alpha=float(alpha_i),
                aspiration=float(aspiration_i),
                initial_policy=int(policy_i),
            )
            self.node_to_agent[int(node_id)] = agent

        # Utilidad inicial
        self.agents.do("update_utility")

        # -------------------------------------------------------------
        # Recolección de datos
        # -------------------------------------------------------------
        self.datacollector = DataCollector(
            model_reporters={
                "ImplementedPolicy": lambda m: m.implemented_policy(),
                "AverageUtility": lambda m: m.average_utility(),
                "AverageAspiration": lambda m: m.average_aspiration(),
                "DissatisfactionRate": lambda m: m.dissatisfaction_rate(),
                "ConsensusLevel": lambda m: m.consensus_level(),
                "EffectiveNumberPolicies": lambda m: m.effective_number_of_policies(),
            },
            agent_reporters={
                "Node": "node_id",
                "Policy": "policy",
                "Utility": "utility",
                "Aspiration": "aspiration",
                "Alpha": "alpha",
            },
        )

        self.datacollector.collect(self)

    # -----------------------------------------------------------------
    # Métricas agregadas
    # -----------------------------------------------------------------
    def policy_counts(self) -> Counter:
        return Counter(agent.policy for agent in self.agents)

    def implemented_policy(self) -> int:
        counts = self.policy_counts()
        max_count = max(counts.values())
        tied = [p for p, cnt in counts.items() if cnt == max_count]

        if len(tied) == 1:
            return tied[0]

        avg_u = {}
        for p in tied:
            vals = [agent.utility for agent in self.agents if agent.policy == p]
            avg_u[p] = float(np.mean(vals)) if vals else -np.inf

        return sorted(tied, key=lambda p: (-avg_u[p], p))[0]

    def average_utility(self) -> float:
        return float(np.mean([agent.utility for agent in self.agents]))

    def average_aspiration(self) -> float:
        return float(np.mean([agent.aspiration for agent in self.agents]))

    def dissatisfaction_rate(self) -> float:
        unsatisfied = [agent.utility < agent.aspiration for agent in self.agents]
        return float(np.mean(unsatisfied))

    def consensus_level(self) -> float:
        counts = self.policy_counts()
        return max(counts.values()) / self.n_agents

    def effective_number_of_policies(self) -> float:
        counts = self.policy_counts()
        shares = np.array(
            [counts.get(p, 0) / self.n_agents for p in self.policies],
            dtype=float
        )
        denom = np.sum(shares ** 2)
        return float(1.0 / denom) if denom > 0 else 0.0

    # -----------------------------------------------------------------
    # Paso temporal
    # -----------------------------------------------------------------
    def step(self) -> None:
        self.agents.do("compute_next_state")
        self.agents.do("advance")
        self.agents.do("update_utility")
        self.datacollector.collect(self)


# ============================================================
# 3. PARÁMETROS FIJOS DEL ENTORNO DE POLÍTICAS
# ============================================================

FIXED_POLICY_QUALITY = [0.90, 0.72, 0.84, 0.61, 0.55, 0.88, 0.79, 0.86, 0.50, 0.67]
FIXED_POLICY_COST    = [0.30, 0.25, 0.35, 0.20, 0.15, 0.40, 0.30, 0.45, 0.10, 0.22]


# ============================================================
# 4. CONSTRUCCIÓN DEL DISEÑO FACTORIAL
# ============================================================

def build_factorial_design(
    repetitions: int = 50,
    beta_values=(0.2, 0.8, 1.4),
    rho_values=(0.05, 0.20, 0.60),
    max_checks_values=(1, 5, 10),
    gamma: float = 0.40,
    avg_degree: int = 6,
    rewire_prob: float = 0.10,
    alpha_mean: float = 1.00,
    alpha_sd: float = 0.10,
    aspiration_mean: float = 0.55,
    aspiration_sd: float = 0.08,
    master_seed: int = 20260318,
) -> pd.DataFrame:
    """
    Construye la matriz del diseño factorial reducido.
    """
    seed_rng = np.random.default_rng(master_seed)
    rows = []
    run_id = 0

    for beta, rho, max_checks in product(beta_values, rho_values, max_checks_values):
        condition_id = f"b{beta}_r{rho}_m{max_checks}"

        for rep in range(1, repetitions + 1):
            run_id += 1
            rows.append(
                {
                    "run_id": run_id,
                    "condition_id": condition_id,
                    "replication": rep,
                    "n_agents": 100,
                    "num_policies": 10,
                    "avg_degree": avg_degree,
                    "rewire_prob": rewire_prob,
                    "beta": beta,
                    "gamma": gamma,
                    "rho": rho,
                    "alpha_mean": alpha_mean,
                    "alpha_sd": alpha_sd,
                    "aspiration_mean": aspiration_mean,
                    "aspiration_sd": aspiration_sd,
                    "max_policy_checks": max_checks,
                    "rng": int(seed_rng.integers(0, np.iinfo(np.int32).max)),
                }
            )

    return pd.DataFrame(rows)


# ============================================================
# 5. FUNCIONES AUXILIARES
# ============================================================

def compute_entropy_from_counts(counts: dict, num_agents: int) -> float:
    probs = np.array([v / num_agents for v in counts.values() if v > 0], dtype=float)
    if len(probs) == 0:
        return 0.0
    return float(-np.sum(probs * np.log(probs)))

def policy_signature(model) -> tuple:
    counts = model.policy_counts()
    return tuple(counts.get(p, 0) for p in model.policies)

def collect_model_state(model, run_info: dict, step: int) -> dict:
    return {
        "run_id": run_info["run_id"],
        "condition_id": run_info["condition_id"],
        "replication": run_info["replication"],
        "Step": step,
        "ImplementedPolicy": model.implemented_policy(),
        "AverageUtility": model.average_utility(),
        "AverageAspiration": model.average_aspiration(),
        "DissatisfactionRate": model.dissatisfaction_rate(),
        "ConsensusLevel": model.consensus_level(),
        "EffectiveNumberPolicies": model.effective_number_of_policies(),
    }

def collect_policy_counts(model, run_info: dict, step: int) -> list[dict]:
    counts = model.policy_counts()
    rows = []

    for p in model.policies:
        rows.append(
            {
                "run_id": run_info["run_id"],
                "condition_id": run_info["condition_id"],
                "replication": run_info["replication"],
                "Step": step,
                "Policy": p,
                "Count": counts.get(p, 0),
                "Share": counts.get(p, 0) / model.n_agents,
            }
        )
    return rows

def collect_final_agent_states(model, run_info: dict, final_step: int) -> list[dict]:
    rows = []
    for agent in model.agents:
        rows.append(
            {
                "run_id": run_info["run_id"],
                "condition_id": run_info["condition_id"],
                "replication": run_info["replication"],
                "FinalStep": final_step,
                "Node": agent.node_id,
                "Policy": agent.policy,
                "Utility": agent.utility,
                "Aspiration": agent.aspiration,
                "Alpha": agent.alpha,
            }
        )
    return rows


# ============================================================
# 6. UNA CORRIDA EXPERIMENTAL
# ============================================================

def run_one_experiment(
    run_info: dict,
    max_steps: int = 100,
    stability_window: int = 5,
    policy_quality=None,
    policy_cost=None,
):
    if policy_quality is None:
        policy_quality = FIXED_POLICY_QUALITY
    if policy_cost is None:
        policy_cost = FIXED_POLICY_COST

    # max_policy_checks puede venir como np.int64 o similar
    max_checks_value = run_info["max_policy_checks"]
    if pd.isna(max_checks_value):
        max_checks_value = None
    else:
        max_checks_value = int(max_checks_value)

    model = PolicyDecisionModel(
        n_agents=int(run_info["n_agents"]),
        num_policies=int(run_info["num_policies"]),
        avg_degree=int(run_info["avg_degree"]),
        rewire_prob=float(run_info["rewire_prob"]),
        beta=float(run_info["beta"]),
        gamma=float(run_info["gamma"]),
        rho=float(run_info["rho"]),
        alpha_mean=float(run_info["alpha_mean"]),
        alpha_sd=float(run_info["alpha_sd"]),
        aspiration_mean=float(run_info["aspiration_mean"]),
        aspiration_sd=float(run_info["aspiration_sd"]),
        max_policy_checks=max_checks_value,
        policy_quality=policy_quality,
        policy_cost=policy_cost,
        rng=int(run_info["rng"]),
    )

    model_rows = []
    policy_rows = []

    # Estado inicial
    step = 0
    model_rows.append(collect_model_state(model, run_info, step))
    policy_rows.extend(collect_policy_counts(model, run_info, step))

    signatures = [policy_signature(model)]
    stability_step = None

    # Dinámica temporal
    for step in range(1, max_steps + 1):
        model.step()

        model_rows.append(collect_model_state(model, run_info, step))
        policy_rows.extend(collect_policy_counts(model, run_info, step))

        signatures.append(policy_signature(model))

        if len(signatures) >= stability_window:
            recent = signatures[-stability_window:]
            if len(set(recent)) == 1:
                stability_step = step
                break

    final_step = step
    final_counts = model.policy_counts()
    entropy_final = compute_entropy_from_counts(final_counts, model.n_agents)

    summary = {
        **run_info,
        "FinalStep": final_step,
        "Stabilized": stability_step is not None,
        "StabilityStep": stability_step if stability_step is not None else np.nan,
        "FinalImplementedPolicy": model.implemented_policy(),
        "FinalAverageUtility": model.average_utility(),
        "FinalAverageAspiration": model.average_aspiration(),
        "FinalDissatisfactionRate": model.dissatisfaction_rate(),
        "FinalConsensusLevel": model.consensus_level(),
        "FinalEffectiveNumberPolicies": model.effective_number_of_policies(),
        "FinalEntropyPolicies": entropy_final,
    }

    for p in model.policies:
        summary[f"CountPolicy_{p}"] = final_counts.get(p, 0)
        summary[f"SharePolicy_{p}"] = final_counts.get(p, 0) / model.n_agents

    final_agent_rows = collect_final_agent_states(model, run_info, final_step)

    return summary, model_rows, policy_rows, final_agent_rows


# ============================================================
# 7. CORRER TODO EL DISEÑO FACTORIAL Y EXPORTAR CSV
# ============================================================

def run_factorial_experiment_suite(
    design_df: pd.DataFrame,
    output_dir: str = "resultados_mesa_factorial",
    max_steps: int = 100,
    stability_window: int = 5,
    policy_quality=None,
    policy_cost=None,
    progress_every: int = 25,
):
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    # Guardar matriz del diseño
    design_df.to_csv(
        output_path / "design_matrix_factorial.csv",
        index=False,
        encoding="utf-8-sig"
    )

    summary_records = []
    model_records = []
    policy_records = []
    final_agent_records = []

    total_runs = len(design_df)

    for i, row in enumerate(design_df.to_dict(orient="records"), start=1):
        summary, model_rows, policy_rows, agent_rows = run_one_experiment(
            row,
            max_steps=max_steps,
            stability_window=stability_window,
            policy_quality=policy_quality,
            policy_cost=policy_cost,
        )

        summary_records.append(summary)
        model_records.extend(model_rows)
        policy_records.extend(policy_rows)
        final_agent_records.extend(agent_rows)

        if i % progress_every == 0 or i == total_runs:
            print(f"Corridas completadas: {i}/{total_runs}")

    df_summary = pd.DataFrame(summary_records)
    df_model = pd.DataFrame(model_records)
    df_policy = pd.DataFrame(policy_records)
    df_agents_final = pd.DataFrame(final_agent_records)

    df_summary.to_csv(
        output_path / "summary_final_factorial.csv",
        index=False,
        encoding="utf-8-sig"
    )
    df_model.to_csv(
        output_path / "model_trajectories_factorial.csv",
        index=False,
        encoding="utf-8-sig"
    )
    df_policy.to_csv(
        output_path / "policy_counts_by_step_factorial.csv",
        index=False,
        encoding="utf-8-sig"
    )
    df_agents_final.to_csv(
        output_path / "agent_final_states_factorial.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("\nExportación finalizada.")
    print(f"Carpeta de salida: {output_path.resolve()}")

    return {
        "design": design_df,
        "summary": df_summary,
        "model_trajectories": df_model,
        "policy_counts_by_step": df_policy,
        "agent_final_states": df_agents_final,
    }


# ============================================================
# 8. EJECUCIÓN
# ============================================================

# PRUEBA CORTA: ponga 2 o 3 antes de lanzar el experimento grande
design_factorial = build_factorial_design(
    repetitions=50,                    # cambie a 50 para el experimento completo
    beta_values=(0.2, 0.8, 1.4),
    rho_values=(0.05, 0.20, 0.60),
    max_checks_values=(1, 5, 10),
    gamma=0.40,
    avg_degree=6,
    rewire_prob=0.10,
    alpha_mean=1.00,
    alpha_sd=0.10,
    aspiration_mean=0.55,
    aspiration_sd=0.08,
    master_seed=20260318,
)

print("Número total de corridas:", len(design_factorial))
print(design_factorial.head())

results_factorial = run_factorial_experiment_suite(
    design_df=design_factorial,
    output_dir="resultados_mesa_factorial",
    max_steps=100,
    stability_window=5,
    policy_quality=FIXED_POLICY_QUALITY,
    policy_cost=FIXED_POLICY_COST,
    progress_every=10,
)

summary_by_condition = (
    results_factorial["summary"]
    .groupby("condition_id", as_index=False)
    .agg(
        MeanConsensus=("FinalConsensusLevel", "mean"),
        SDConsensus=("FinalConsensusLevel", "std"),
        MeanENP=("FinalEffectiveNumberPolicies", "mean"),
        SDENP=("FinalEffectiveNumberPolicies", "std"),
        MeanUtility=("FinalAverageUtility", "mean"),
        MeanDissatisfaction=("FinalDissatisfactionRate", "mean"),
        MeanStabilityStep=("StabilityStep", "mean"),
    )
)

print("\n=== Resumen por condición experimental ===")
print(summary_by_condition.head(10))

Número total de corridas: 1350
   run_id   condition_id  replication  n_agents  num_policies  avg_degree  \
0       1  b0.2_r0.05_m1            1       100            10           6   
1       2  b0.2_r0.05_m1            2       100            10           6   
2       3  b0.2_r0.05_m1            3       100            10           6   
3       4  b0.2_r0.05_m1            4       100            10           6   
4       5  b0.2_r0.05_m1            5       100            10           6   

   rewire_prob  beta  gamma   rho  alpha_mean  alpha_sd  aspiration_mean  \
0          0.1   0.2    0.4  0.05         1.0       0.1             0.55   
1          0.1   0.2    0.4  0.05         1.0       0.1             0.55   
2          0.1   0.2    0.4  0.05         1.0       0.1             0.55   
3          0.1   0.2    0.4  0.05         1.0       0.1             0.55   
4          0.1   0.2    0.4  0.05         1.0       0.1             0.55   

   aspiration_sd  max_policy_checks         rng  